In [2]:
import torch
import torchvision
print(torch.__version__)
print(torchvision.__version__)

2.3.1+cu121
0.18.1+cu121


In [3]:
from torch import nn
from torchvision import transforms
!pip install -q torchinfo
from torchinfo import summary

!git clone https://github.com/mrdbourke/pytorch-deep-learning
!mv pytorch-deep-learning/going_modular .
!mv pytorch-deep-learning/helper_functions.py .
!rm -rf pytorch-deep-learning
from going_modular.going_modular import data_setup, engine
from helper_functions import download_data, set_seeds, plot_loss_curves

Cloning into 'pytorch-deep-learning'...
remote: Enumerating objects: 4177, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 4177 (delta 61), reused 105 (delta 39), pack-reused 4035 (from 1)
Receiving objects: 100% (4177/4177), 651.42 MiB | 33.33 MiB/s, done.
Resolving deltas: 100% (2433/2433), done.
Updating files: 100% (248/248), done.


In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cpu'

In [5]:
data_20_percent_path = download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip",
                                     destination="pizza_steak_sushi_20_percent")

data_20_percent_path

[INFO] Did not find data/pizza_steak_sushi_20_percent directory, creating one...
[INFO] Downloading pizza_steak_sushi_20_percent.zip from https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip...
[INFO] Unzipping pizza_steak_sushi_20_percent.zip data...


PosixPath('data/pizza_steak_sushi_20_percent')

In [6]:
train_dir = data_20_percent_path / "train"
test_dir = data_20_percent_path / "test"

In [7]:
effnetb2_weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT

effnetb2_transforms=effnetb2_weights.transforms()

effnetb2=torchvision.models.efficientnet_b2(weights=effnetb2_weights)

for param in effnetb2.parameters():
  param.requires_grad=False

Downloading: "https://download.pytorch.org/models/efficientnet_b2_rwightman-c35c1473.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b2_rwightman-c35c1473.pth
100%|██████████| 35.2M/35.2M [00:01<00:00, 35.7MB/s]


In [8]:
effnetb2.classifier

Sequential(
  (0): Dropout(p=0.3, inplace=True)
  (1): Linear(in_features=1408, out_features=1000, bias=True)
)

In [9]:
effnetb2.classifier=nn.Sequential(
    nn.Dropout(p=0.3,inplace=True),
    nn.Linear(in_features=1408,
              out_features=3)
)

In [10]:
def create_effnetb2_model(num_classes:int=3,seed:int=42):

  weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT
  transforms=weights.transforms()
  model=torchvision.models.efficientnet_b2(weights=weights)

  for param in model.parameters():
    param.requires_grad=False


  torch.manual_seed(seed)
  model.classifier=nn.Sequential(
    nn.Dropout(p=0.3,inplace=True),
    nn.Linear(in_features=1408,
              out_features=3)
)

  return model,transforms

In [11]:
effnetb2,effnetb2_transforms=create_effnetb2_model(3,42)

In [13]:
from torchinfo import summary

summary(effnetb2,
        input_size=(1,3,224,224),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])


Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [1, 3, 224, 224]     [1, 3]               --                   Partial
├─Sequential (features)                                      [1, 3, 224, 224]     [1, 1408, 7, 7]      --                   False
│    └─Conv2dNormActivation (0)                              [1, 3, 224, 224]     [1, 32, 112, 112]    --                   False
│    │    └─Conv2d (0)                                       [1, 3, 224, 224]     [1, 32, 112, 112]    (864)                False
│    │    └─BatchNorm2d (1)                                  [1, 32, 112, 112]    [1, 32, 112, 112]    (64)                 False
│    │    └─SiLU (2)                                         [1, 32, 112, 112]    [1, 32, 112, 112]    --                   --
│    └─Sequential (1)                                        [1, 32, 112, 112]    [1, 1

In [14]:
from going_modular.going_modular import data_setup
train_dataloader_effnetb2, test_dataloader_effnetb2, class_names = data_setup.create_dataloaders(train_dir=train_dir,
                                                                                                 test_dir=test_dir,
                                                                                                 transform=effnetb2_transforms,
                                                                                                 batch_size=32)


In [15]:
from going_modular.going_modular import engine

optimizer=torch.optim.Adam(params=effnetb2.parameters(),
                           lr=1e-3)

loss_fn=torch.nn.CrossEntropyLoss()

set_seeds()

effnetb2_results=engine.train(model=effnetb2,
                              train_dataloader=train_dataloader_effnetb2,
                              test_dataloader=test_dataloader_effnetb2,
                              epochs=10,
                              optimizer=optimizer,
                              loss_fn=loss_fn,
                              device=device)

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch: 1 | train_loss: 0.9794 | train_acc: 0.5708 | test_loss: 0.7390 | test_acc: 0.9284
Epoch: 2 | train_loss: 0.6835 | train_acc: 0.8771 | test_loss: 0.5929 | test_acc: 0.9347
Epoch: 3 | train_loss: 0.5640 | train_acc: 0.8958 | test_loss: 0.4816 | test_acc: 0.9409
Epoch: 4 | train_loss: 0.4716 | train_acc: 0.8729 | test_loss: 0.4183 | test_acc: 0.9597
Epoch: 5 | train_loss: 0.4308 | train_acc: 0.8729 | test_loss: 0.3723 | test_acc: 0.9659
Epoch: 6 | train_loss: 0.3738 | train_acc: 0.8979 | test_loss: 0.3456 | test_acc: 0.9506
Epoch: 7 | train_loss: 0.3373 | train_acc: 0.9062 | test_loss: 0.3132 | test_acc: 0.9568
Epoch: 8 | train_loss: 0.3248 | train_acc: 0.9250 | test_loss: 0.3009 | test_acc: 0.9597
Epoch: 9 | train_loss: 0.3671 | train_acc: 0.8562 | test_loss: 0.2747 | test_acc: 0.9534
Epoch: 10 | train_loss: 0.2564 | train_acc: 0.9458 | test_loss: 0.2649 | test_acc: 0.9597


In [26]:
from going_modular.going_modular import utils
utils.save_model(model=effnetb2,
                 target_dir='models',
                 model_name="effnetb2_final_food_vision_model.pth")

[INFO] Saving model to: models/effnetb2_final_food_vision_model.pth


In [16]:
!pip install -q gradio
import gradio as gr
print(gr.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.9/141.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 7.6 MB/s eta 0:00:00
4.42.0


In [21]:
from typing import Tuple, Dict
from PIL import Image
from timeit import default_timer as timer
from tqdm.auto import tqdm
from typing import List, Dict

def predict(img)->Tuple[Dict,float]:

  start_timer=timer()
  img=effnetb2_transforms(img).unsqueeze(0)

  effnetb2.eval()
  with torch.inference_mode():
    pred_probs=torch.softmax(effnetb2(img),dim=1)

  pred_labels_and_probs={class_names[i]:float(pred_probs[0][i])for i in range(len(class_names))}

  pred_timer=round(timer() - start_timer,5)

  return pred_labels_and_probs,pred_timer

In [23]:
import random
from PIL import Image
from pathlib import Path


test_data_paths=list(Path(test_dir).glob("*/*.jpg"))

random_image_path = random.sample(test_data_paths,k=1)[0]

image=Image.open(random_image_path)

pred_dict,pred_timer=predict(img=image)
print(f"Prediction label and probability dictionary: \n{pred_dict}")
print(f"Prediction time: {pred_timer} seconds")


Prediction label and probability dictionary: 
{'pizza': 0.14041347801685333, 'steak': 0.07695005089044571, 'sushi': 0.7826365232467651}
Prediction time: 0.13477 seconds


In [31]:
example_list = [[str(filepath) for filepath in random.sample(test_data_paths,k=5)]]
example_list

[['data/pizza_steak_sushi_20_percent/test/pizza/2582289.jpg',
  'data/pizza_steak_sushi_20_percent/test/pizza/296426.jpg',
  'data/pizza_steak_sushi_20_percent/test/pizza/2398925.jpg',
  'data/pizza_steak_sushi_20_percent/test/sushi/911808.jpg',
  'data/pizza_steak_sushi_20_percent/test/steak/1848936.jpg']]

In [32]:
from operator import truediv
import gradio as gr

title = "Food Vision 🍕🥩🍣 "
description = "A computer vision model for classifying food images, built using EfficientNet-B2 "
article = "EfficientNet-B2 powered model for accurate food image classification. Upload a food image to see predicted categories and probabilities."
demo = gr.Interface(fn=predict,
                    inputs=gr.Image(type='pil'),
                    outputs=[gr.Label(num_top_classes=3,label="Predictions"),
                             gr.Number(label="Prediction time (s) ")],
                    examples=example_list,
                    title=title,
                    description=description,
                    article=article)

demo.launch(debug=False,
            share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://1d5ec4440aa9afd910.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [36]:
import shutil
from pathlib import Path

foodvision_path = Path("demos/foodvision/")

foodvision_path.mkdir(parents=True,
                      exist_ok=True)
!ls demos/foodvision/

In [37]:
import shutil
from pathlib import Path

foodvision_examples_path = foodvision_path/"examples"
foodvision_examples_path.mkdir(parents=True,exist_ok=True)

foodvision_examples=[Path('data/pizza_steak_sushi_20_percent/test/pizza/2398925.jpg'),
                          Path('data/pizza_steak_sushi_20_percent/test/sushi/911808.jpg'),
                          Path('data/pizza_steak_sushi_20_percent/test/steak/1848936.jpg')]

for example in foodvision_examples:
  destination = foodvision_examples_path/example.name
  shutil.copy2(src=example,dst=destination)


In [38]:
import shutil

effnetb2_foodvision_model_path="models/effnetb2_final_food_vision_model.pth"

effnetb2_foodvision_model_destination=foodvision_path/effnetb2_foodvision_model_path.split("/")[1]

shutil.move(src=effnetb2_foodvision_model_path,
            dst=effnetb2_foodvision_model_destination)

PosixPath('demos/foodvision/effnetb2_final_food_vision_model.pth')

In [39]:
%%writefile demos/foodvision/model.py
import torch
import torchvision

from torch import nn


def create_effnetb2_model(num_classes:int=3,
                          seed:int=42):
    """Creates an EfficientNetB2 feature extractor model and transforms.

    Args:
        num_classes (int, optional): number of classes in the classifier head.
            Defaults to 3.
        seed (int, optional): random seed value. Defaults to 42.

    Returns:
        model (torch.nn.Module): EffNetB2 feature extractor model.
        transforms (torchvision.transforms): EffNetB2 image transforms.
    """
    # Create EffNetB2 pretrained weights, transforms and model
    weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT
    transforms = weights.transforms()
    model = torchvision.models.efficientnet_b2(weights=weights)

    # Freeze all layers in base model
    for param in model.parameters():
        param.requires_grad = False

    # Change classifier head with random seed for reproducibility
    torch.manual_seed(seed)
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features=1408, out_features=num_classes),
    )

    return model, transforms

Writing demos/foodvision/model.py


In [49]:
%%writefile demos/foodvision/app.py

import gradio as gr
import os
import torch

from model import create_effnetb2_model
from timeit import default_timer as timer
from typing import Tuple,Dict

class_names=["pizza","steak","sushi"]

effnetb2,effnetb2_transforms=create_effnetb2_model(
    num_classes=3
)

effnetb2.load_state_dict(
    torch.load(
        f="effnetb2_final_food_vision_model.pth",
        map_location=torch.device("cpu"),
    )
)

def predict(img)->Tuple[Dict,float]:

  start_timer=timer()
  img=effnetb2_transforms(img).unsqueeze(0)

  effnetb2.eval()
  with torch.inference_mode():
    pred_probs=torch.softmax(effnetb2(img),dim=1)

  pred_labels_and_probs={class_names[i]:float(pred_probs[0][i])for i in range(len(class_names))}

  pred_timer=round(timer() - start_timer,5)

  return pred_labels_and_probs,pred_timer

title = "Food Vision 🍕🥩🍣 "
description = "A computer vision model for classifying food images, built using EfficientNet-B2 "
article = "EfficientNet-B2 powered model for accurate food image classification. Upload a food image to see predicted categories and probabilities."

examples=[["examples/"+ example] for example in os.listdir("examples")]

demo = gr.Interface(fn=predict,
                    inputs=gr.Image(type='pil'),
                    outputs=[gr.Label(num_top_classes=3,label="Predictions"),
                             gr.Number(label="Prediction time (s) ")],
                    examples=example_list,
                    title=title,
                    description=description,
                    article=article)

demo.launch()

Overwriting demos/foodvision/app.py


In [41]:
print(torch.__version__)
print(torchvision.__version__)
print(gr.__version__)

2.3.1+cu121
0.18.1+cu121
4.42.0


In [42]:
%%writefile demos/foodvision/requirements.txt

torch==2.3.1
torchvision==0.18.1
gradio==4.42.0

Writing demos/foodvision/requirements.txt


In [43]:
!ls demos/foodvision

app.py	effnetb2_final_food_vision_model.pth  examples	model.py  requirements.txt


In [50]:
!cd demos/foodvision && zip -r ../foodvision.zip * -x "*.pyc" "*.ipynb" "*__pycache__*" "*ipynb_checkpoints*"

try:
    from google.colab import files
    files.download("demos/foodvision.zip")
except:
    print("Not running in Google Colab, can't use google.colab.files.download(), please manually download.")

updating: app.py (deflated 53%)
updating: effnetb2_final_food_vision_model.pth (deflated 8%)
updating: examples/ (stored 0%)
updating: examples/1848936.jpg (deflated 1%)
updating: examples/911808.jpg (deflated 0%)
updating: examples/2398925.jpg (deflated 0%)
updating: model.py (deflated 56%)
updating: requirements.txt (deflated 6%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>